# 02 · Transformaciones — Bronze → Silver
### Proyecto ETL FIFA 21 · Jorge Amat · David Plaza

**Requisito previo:** haber ejecutado `carga_delta.ipynb` (existe `fifa_catalog.bronze.fifa_21_delta`).

**Qué hace este notebook:**
1. Calcula el **Data Quality Report** sobre Bronze (completitud, unicidad, validez, consistencia, exactitud, actualidad).
2. Aplica la limpieza y las transformaciones: tipos de altura/peso, variables monetarias, estrellas, columna `team`, posiciones, scores derivados, corrección de stats de portero, deduplicado por `id`.
3. Vuelve a calcular el Data Quality Report sobre el resultado limpio, para comparar la mejora.
4. Escribe la tabla final en `fifa_catalog.silver.df_silver` y la optimiza (`OPTIMIZE ... ZORDER`, `VACUUM`).

**Siguiente notebook:** `MODELLING.ipynb`

In [0]:
from pyspark.sql import functions as f
from pyspark.sql.functions import col
from functools import reduce
from pyspark.sql import types as T
from pyspark.sql.functions import split, size, lit, current_timestamp
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_extract, regexp_replace,trim

spark = SparkSession.builder \
    .appName("MiSesionSpark") \
    .getOrCreate()

print("Spark iniciado ✔️")

Spark iniciado ✔️


In [0]:
#Visualizamos la tabla
df_bronze = spark.table("fifa_catalog.bronze.fifa_21_delta")
display(df_bronze.limit(20))

photourl,longname,playerurl,nationality,positions,name,age,_ova,pot,team,id,height,weight,foot,bov,bp,growth,joined,loan_date_end,value,wage,release_clause,attacking,crossing,finishing,heading_accuracy,short_passing,volleys,skill,dribbling,curve,fk_accuracy,long_passing,ball_control,movement,acceleration,sprint_speed,agility,reactions,balance,power,shot_power,jumping,stamina,strength,long_shots,mentality,aggression,interceptions,positioning,vision,penalties,composure,defending,marking,standing_tackle,sliding_tackle,goalkeeping,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes,total_stats,base_stats,w_f,sm,a_w,d_w,ir,pac,sho,pas,dri,def,phy,hits
https://cdn.sofifa.com/players/158/023/21_60.png,Lionel Messi,http://sofifa.com/player/158023/lionel-messi/210005/,Argentina,RW ST CF,L. Messi,33,93,93,FC Barcelona 2004 ~ 2021,158023,"5'7""",159lbs,Left,93,RW,0,"Jul 1, 2004",N/A,€67.5M,€560K,€138.4M,429,85,95,70,91,88,470,96,93,94,91,96,451,91,80,91,94,95,389,86,68,72,69,94,347,44,40,93,95,75,96,91,32,35,24,54,6,11,15,14,8,2231,466,4 ★,4★,Medium,Low,5 ★,85,92,91,95,38,65,372
https://cdn.sofifa.com/players/020/801/21_60.png,C. Ronaldo dos Santos Aveiro,http://sofifa.com/player/20801/c-ronaldo-dos-santos-aveiro/210005/,Portugal,ST LW,Cristiano Ronaldo,35,92,92,Juventus 2018 ~ 2022,20801,"6'2""",183lbs,Right,92,ST,0,"Jul 10, 2018",N/A,€46M,€220K,€75.9M,437,84,95,90,82,86,414,88,81,76,77,92,431,87,91,87,95,71,444,94,95,84,78,93,353,63,29,95,82,84,95,84,28,32,24,58,7,11,15,14,11,2221,464,4 ★,5★,High,Low,5 ★,89,93,81,89,35,77,344
https://cdn.sofifa.com/players/200/389/21_60.png,Jan Oblak,http://sofifa.com/player/200389/jan-oblak/210005/,Slovenia,GK,J. Oblak,27,91,93,Atlético Madrid 2014 ~ 2023,200389,"6'2""",192lbs,Right,91,GK,2,"Jul 16, 2014",N/A,€75M,€125K,€159.4M,95,13,11,15,43,13,109,12,13,14,40,30,307,43,60,67,88,49,268,59,78,41,78,12,140,34,19,11,65,11,68,57,27,12,18,437,87,92,78,90,90,1413,489,3 ★,1★,Medium,Medium,3 ★,87,92,78,90,52,90,86
https://cdn.sofifa.com/players/192/985/21_60.png,Kevin De Bruyne,http://sofifa.com/player/192985/kevin-de-bruyne/210005/,Belgium,CAM CM,K. De Bruyne,29,91,91,Manchester City 2015 ~ 2023,192985,"5'11""",154lbs,Right,91,CAM,0,"Aug 30, 2015",N/A,€87M,€370K,€161M,407,94,82,55,94,82,441,88,85,83,93,92,398,77,76,78,91,76,408,91,63,89,74,91,408,76,66,88,94,84,91,186,68,65,53,56,15,13,5,10,13,2304,485,5 ★,4★,High,High,4 ★,76,86,93,88,64,78,163
https://cdn.sofifa.com/players/190/871/21_60.png,Neymar da Silva Santos Jr.,http://sofifa.com/player/190871/neymar-da-silva-santos-jr/210005/,Brazil,LW CAM,Neymar Jr,28,91,91,Paris Saint-Germain 2017 ~ 2022,190871,"5'9""",150lbs,Right,91,LW,0,"Aug 3, 2017",N/A,€90M,€270K,€166.5M,408,85,87,62,87,87,448,95,88,89,81,95,453,94,89,96,91,83,357,80,62,81,50,84,356,51,36,87,90,92,93,94,35,30,29,59,9,9,15,15,11,2175,451,5 ★,5★,High,Medium,5 ★,91,85,86,94,36,59,273
https://cdn.sofifa.com/players/188/545/21_60.png,Robert Lewandowski,http://sofifa.com/player/188545/robert-lewandowski/210005/,Poland,ST,R. Lewandowski,31,91,91,FC Bayern München 2014 ~ 2023,188545,"6'0""",176lbs,Right,91,ST,0,"Jul 1, 2014",N/A,€80M,€240K,€132M,423,71,94,85,84,89,407,85,79,85,70,88,407,77,78,77,93,82,420,89,84,76,86,85,391,81,49,94,79,88,88,96,35,42,19,51,15,6,12,8,10,2195,457,4 ★,4★,High,Medium,4 ★,78,91,78,85,43,82,182
https://cdn.sofifa.com/players/231/747/21_60.png,Kylian Mbappé,http://sofifa.com/player/231747/kylian-mbappe/210005/,France,ST LW RW,K. Mbappé,21,90,95,Paris Saint-Germain 2018 ~ 2022,231747,"5'10""",161lbs,Right,91,ST,5,"Jul 1, 2018",N/A,€105.5M,€160K,€203.1M,408,78,91,73,83,83,394,92,79,63,70,90,458,96,96,92,92,82,404,86,77,86,76,79,341,62,38,91,80,70,84,100,34,34,32,42,13,5,7,11,6,2147,466,4 ★,5★,High,Low,3 ★,96,86,78,91,39,76,646
https://cdn.sofifa.com/players/212/831/21_60.png,Alisson Ramses Becker,http://sofifa.com/player/212831/alisson-ramses-becker/210005/,Brazil,GK,Alisson,27,90,91,Liverpool 2018 ~ 2024,212831,"6'3""",201lbs,Right,90,GK,1,"Jul 19, 2018",N/A,€62.5M,€

In [0]:
df = df_bronze  # alias


# ============================================================
# 1) COMPLETITUD (no nulos)
# ============================================================

nulls = df.select([
    f.sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).first()

total_nulls = sum(nulls[c] for c in nulls.__fields__)
total_cells = df.count() * len(df.columns)

completitud = 1 - (total_nulls / total_cells)


# ============================================================
# 2) UNICIDAD (sin duplicados)
# ============================================================

total_rows = df.count()
unique_rows = df.dropDuplicates().count()

unicidad = unique_rows / total_rows


# ============================================================
# 3) VALIDEZ (checks moderados y realistas)
# ============================================================

# --- Edad razonable ---
invalid_age = df.filter((col("age") < 15) | (col("age") > 55)).count()

# --- Height válido: debe empezar con números ---
invalid_height = df.filter(
    ~col("height").rlike("^[0-9]{2,3}.*$")
).count()

# --- Weight válido: debe empezar con números ---
invalid_weight = df.filter(
    ~col("weight").rlike("^[0-9]{2,3}.*$")
).count()

# --- Growth razonable (0 a 25 es típico FIFA) ---
invalid_growth = df.filter(
    (col("growth") < 0) | (col("growth") > 25)
).count()

# --- Value / Wage / Release Clause ---
money_columns = ["value", "wage", "release_clause"]

invalid_money = df.filter(
    reduce(
        lambda a, b: a | b,
        [col(c).rlike("^€[0-9MK]+$") for c in money_columns]
    )
).count()

# TOTAL inválidos moderados
invalid_total = (
    invalid_age +
    invalid_height +
    invalid_weight +
    invalid_growth +
    invalid_money
)

# límite para evitar resultados absurdos
invalid_total = min(invalid_total, total_rows)

validez = 1 - (invalid_total / total_rows)
validez = max(validez, 0)   # evitar negativos


# ============================================================
# 4) CONSISTENCIA (coherencia interna)
# ============================================================

inconsistent_gk = df.filter(
    (~col("bp").contains("GK")) &
    (
        (col("gk_diving") > 0) |
        (col("gk_handling") > 0) |
        (col("gk_kicking") > 0) |
        (col("gk_positioning") > 0) |
        (col("gk_reflexes") > 0)
    )
).count()

consistencia = 1 -(inconsistent_gk / total_rows)


# ============================================================
# 5) EXACTITUD (stats en 0–100)
# ============================================================

stat_columns = [
    'pac', 'sho', 'pas', 'dri', 'def', 'phy',
    'acceleration', 'sprint_speed', 'agility',
    'reactions', 'strength', 'composure'
]

invalid_stats = df.filter(
    reduce(
        lambda a, b: a | b,
        [(col(c) < 0) | (col(c) > 100) for c in stat_columns]
    )
).count()

exactitud = 1 - (invalid_stats / total_rows)


# ============================================================
# 6) ACTUALIDAD (proxy: joined no nulo)
# ============================================================

missing_joined = df.filter(col("joined").isNull()).count()

actualidad = 1 - (missing_joined / total_rows)


# ============================================================
# SCORE GLOBAL
# ============================================================

data_quality_score = (
    completitud +
    unicidad +
    validez +
    consistencia +
    exactitud +
    actualidad
) / 6


# ============================================================
# REPORT FINAL
# ============================================================

print("=== DATA QUALITY REPORT ===")
print(f"Completitud:   {completitud*100:.2f}%")
print(f"Unicidad:      {unicidad*100:.2f}%")
print(f"Validez:       {validez*100:.2f}%")
print(f"Consistencia:  {consistencia*100:.2f}%")
print(f"Exactitud:     {exactitud*100:.2f}%")
print(f"Actualidad:    {actualidad*100:.2f}%")
print("-------------------------------------")
print(f"DATA QUALITY GLOBAL: {data_quality_score*100:.2f}%")


=== DATA QUALITY REPORT ===
Completitud:   100.00%
Unicidad:      99.99%
Validez:       0.00%
Consistencia:  10.93%
Exactitud:     100.00%
Actualidad:    100.00%
-------------------------------------
DATA QUALITY GLOBAL: 68.49%


In [0]:
string_cols = [c for c, t in df_bronze.dtypes if t == 'string']

df_silver = df_bronze
for c in string_cols:
    df_silver = df_silver.withColumn(c, f.regexp_replace(f.col(c), "\n", ""))

In [0]:
# Convertir formato: 5'11" → centímetros directamente en height
# (usamos try_cast-safe: si el formato no matchea, regexp_extract devuelve '' y .cast() explota;
#  con el when->otherwise convertimos ese '' en NULL en vez de romper la celda)
feet = f.regexp_extract(f.col("height"), r"(\d+)'", 1)
inches = f.regexp_extract(f.col("height"), r"'(\d+)", 1)

df_silver = df_silver.withColumn(
    "height",
    (
        f.when(feet != "", feet.cast("float")).otherwise(f.lit(None).cast("float")) * 12 +
        f.when(inches != "", inches.cast("float")).otherwise(f.lit(None).cast("float"))
    ) * 2.54
)

# Redondear
df_silver = df_silver.withColumn("height", f.round(f.col("height"), 1))


In [0]:
#Peso a de lbs a kg
# (mismo motivo que height: si weight viene vacío o sin 'lbs', el cast directo revienta.
#  Convertimos '' en NULL en vez de que falle toda la celda)

peso_kg = f.regexp_replace(f.col("weight"), "lbs", "")

df_silver = df_silver.withColumn(
    "weight",
    f.when(peso_kg != "", peso_kg.cast("float")).otherwise(f.lit(None).cast("float")) * 0.45359237
)

# Redondear a 1 decimal
df_silver = df_silver.withColumn(
    "weight",
    f.round(f.col("weight"), 1)
)


In [0]:
# Limpiamos las columnas con estrellas
star_cols = ["w_f", "sm", "ir"]

for c in star_cols:
    if c in df_silver.columns:
        df_silver = df_silver.withColumn(
            c,
            f.regexp_replace(f.col(c), r"[^0-9]", "").cast("int")
        )

In [0]:
# Quitamos las "M" y "K" de unidades y ponemos el número directamente

money_cols = ["value", "wage", "release_clause"]

for c in money_cols:

    # parte numérica
    num = f.regexp_extract(f.col(c), r"([0-9]+(\.[0-9]+)?)", 1).cast("double")
    
    # sufijo, que puede ser M o K
    suf = f.regexp_extract(f.col(c), r"([MK])", 1)

    df_silver = df_silver.withColumn(
        c,
        f.round(
            f.when(suf == "M", num * 1_000_000)   # millones
             .when(suf == "K", num * 1_000)       # miles
             .otherwise(num)                      # por si acaso viene sin sufijo
        ).cast("long")  # o "int" si prefieres
    )


In [0]:
#Ponemos nombres un poco mas descriptivos. (Los sabemos porque si te metes a la URl del jugador que hay en la tabla se ve lo que significa cada columna)

df_silver = df_silver.withColumnRenamed("w_f", "weak_foot_starss")
df_silver = df_silver.withColumnRenamed("sm", "skills_stars")
df_silver = df_silver.withColumnRenamed("a_w", "atack_contribution")
df_silver = df_silver.withColumnRenamed("d_w", "defense_contribution")
df_silver = df_silver.withColumnRenamed("ir", "international_reputation")
df_silver = df_silver.withColumnRenamed("_ova", "overall_rating")
df_silver = df_silver.withColumnRenamed("pot", "potential")
df_silver = df_silver.withColumnRenamed("bov", "best_overall")
df_silver = df_silver.withColumnRenamed("bp", "best_position")
display(df_silver.limit(20))

photourl,longname,playerurl,nationality,positions,name,age,overall_rating,potential,team,id,height,weight,foot,best_overall,best_position,growth,joined,loan_date_end,value,wage,release_clause,attacking,crossing,finishing,heading_accuracy,short_passing,volleys,skill,dribbling,curve,fk_accuracy,long_passing,ball_control,movement,acceleration,sprint_speed,agility,reactions,balance,power,shot_power,jumping,stamina,strength,long_shots,mentality,aggression,interceptions,positioning,vision,penalties,composure,defending,marking,standing_tackle,sliding_tackle,goalkeeping,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes,total_stats,base_stats,weak_foot_starss,skills_stars,atack_contribution,defense_contribution,international_reputation,pac,sho,pas,dri,def,phy,hits
https://cdn.sofifa.com/players/158/023/21_60.png,Lionel Messi,http://sofifa.com/player/158023/lionel-messi/210005/,Argentina,RW ST CF,L. Messi,33,93,93,FC Barcelona2004 ~ 2021,158023,170.2,72.1,Left,93,RW,0,"Jul 1, 2004",N/A,67500000,560000,138400000,429,85,95,70,91,88,470,96,93,94,91,96,451,91,80,91,94,95,389,86,68,72,69,94,347,44,40,93,95,75,96,91,32,35,24,54,6,11,15,14,8,2231,466,4,4,Medium,Low,5,85,92,91,95,38,65,372
https://cdn.sofifa.com/players/020/801/21_60.png,C. Ronaldo dos Santos Aveiro,http://sofifa.com/player/20801/c-ronaldo-dos-santos-aveiro/210005/,Portugal,ST LW,Cristiano Ronaldo,35,92,92,Juventus2018 ~ 2022,20801,188.0,83.0,Right,92,ST,0,"Jul 10, 2018",N/A,46000000,220000,75900000,437,84,95,90,82,86,414,88,81,76,77,92,431,87,91,87,95,71,444,94,95,84,78,93,353,63,29,95,82,84,95,84,28,32,24,58,7,11,15,14,11,2221,464,4,5,High,Low,5,89,93,81,89,35,77,344
https://cdn.sofifa.com/players/200/389/21_60.png,Jan Oblak,http://sofifa.com/player/200389/jan-oblak/210005/,Slovenia,GK,J. Oblak,27,91,93,Atlético Madrid2014 ~ 2023,200389,188.0,87.1,Right,91,GK,2,"Jul 16, 2014",N/A,75000000,125000,159400000,95,13,11,15,43,13,109,12,13,14,40,30,307,43,60,67,88,49,268,59,78,41,78,12,140,34,19,11,65,11,68,57,27,12,18,437,87,92,78,90,90,1413,489,3,1,Medium,Medium,3,87,92,78,90,52,90,86
https://cdn.sofifa.com/players/192/985/21_60.png,Kevin De Bruyne,http://sofifa.com/player/192985/kevin-de-bruyne/210005/,Belgium,CAM CM,K. De Bruyne,29,91,91,Manchester City2015 ~ 2023,192985,180.3,69.9,Right,91,CAM,0,"Aug 30, 2015",N/A,87000000,370000,161000000,407,94,82,55,94,82,441,88,85,83,93,92,398,77,76,78,91,76,408,91,63,89,74,91,408,76,66,88,94,84,91,186,68,65,53,56,15,13,5,10,13,2304,485,5,4,High,High,4,76,86,93,88,64,78,163
https://cdn.sofifa.com/players/190/871/21_60.png,Neymar da Silva Santos Jr.,http://sofifa.com/player/190871/neymar-da-silva-santos-jr/210005/,Brazil,LW CAM,Neymar Jr,28,91,91,Paris Saint-Germain2017 ~ 2022,190871,175.3,68.0,Right,91,LW,0,"Aug 3, 2017",N/A,90000000,270000,166500000,408,85,87,62,87,87,448,95,88,89,81,95,453,94,89,96,91,83,357,80,62,81,50,84,356,51,36,87,90,92,93,94,35,30,29,59,9,9,15,15,11,2175,451,5,5,High,Medium,5,91,85,86,94,36,59,273
https://cdn.sofifa.com/players/188/545/21_60.png,Robert Lewandowski,http://sofifa.com/player/188545/robert-lewandowski/210005/,Poland,ST,R. Lewandowski,31,91,91,FC Bayern München2014 ~ 2023,188545,182.9,79.8,Right,91,ST,0,"Jul 1, 2014",N/A,80000000,240000,132000000,423,71,94,85,84,89,407,85,79,85,70,88,407,77,78,77,93,82,420,89,84,76,86,85,391,81,49,94,79,88,88,96,35,42,19,51,15,6,12,8,10,2195,457,4,4,High,Medium,4,78,91,78,85,43,82,182
https://cdn.sofifa.com/players/231/747/21_60.png,Kylian Mbappé,http://sofifa.com/player/231747/kylian-mbappe/210005/,France,ST LW RW,K. Mbappé,21,90,95,Paris Saint-Germain2018 ~ 2022,231747,177.8,73.0,Right,91,ST,5,"Jul 1, 2018",N/A,105500000,160000,203100000,408,78,91,73,83,83,394,92,79,63,70,90,458,96,96,92,92,82,404,86,77,86,76,79,341,62,38,91,80,70,84,100,34,34,32,42,13,5,7,11,6,2147,466,4,5,High,Low,3,96,86,78,91,39,76,646
https://cdn.sofifa.com/players/212/831/21_60.png,Alisson Ramses Becker,http://sofifa.com/player/212831/alisson-ramses-becker/210005/,Brazil,GK,Alisson,27,9

In [0]:
#Dividimos la columna de equipo en equipo y contrato

df_silver = df_silver.withColumn("team_name", regexp_extract("team", r'^([^\d]+)', 1))
df_silver = df_silver.withColumn("contract", regexp_extract("team", r'(\d.*)', 1))
df_silver = df_silver.withColumn("team_name", trim(df_silver["team_name"]))
df_silver = df_silver.withColumn("contract", trim(df_silver["contract"]))
df_silver = df_silver.withColumn("contract", regexp_replace(trim(df_silver["contract"]), "~", "-"))
display(df_silver.limit(20))

photourl,longname,playerurl,nationality,positions,name,age,overall_rating,potential,team,id,height,weight,foot,best_overall,best_position,growth,joined,loan_date_end,value,wage,release_clause,attacking,crossing,finishing,heading_accuracy,short_passing,volleys,skill,dribbling,curve,fk_accuracy,long_passing,ball_control,movement,acceleration,sprint_speed,agility,reactions,balance,power,shot_power,jumping,stamina,strength,long_shots,mentality,aggression,interceptions,positioning,vision,penalties,composure,defending,marking,standing_tackle,sliding_tackle,goalkeeping,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes,total_stats,base_stats,weak_foot_starss,skills_stars,atack_contribution,defense_contribution,international_reputation,pac,sho,pas,dri,def,phy,hits,team_name,contract
https://cdn.sofifa.com/players/158/023/21_60.png,Lionel Messi,http://sofifa.com/player/158023/lionel-messi/210005/,Argentina,RW ST CF,L. Messi,33,93,93,FC Barcelona2004 ~ 2021,158023,170.2,72.1,Left,93,RW,0,"Jul 1, 2004",N/A,67500000,560000,138400000,429,85,95,70,91,88,470,96,93,94,91,96,451,91,80,91,94,95,389,86,68,72,69,94,347,44,40,93,95,75,96,91,32,35,24,54,6,11,15,14,8,2231,466,4,4,Medium,Low,5,85,92,91,95,38,65,372,FC Barcelona,2004 - 2021
https://cdn.sofifa.com/players/020/801/21_60.png,C. Ronaldo dos Santos Aveiro,http://sofifa.com/player/20801/c-ronaldo-dos-santos-aveiro/210005/,Portugal,ST LW,Cristiano Ronaldo,35,92,92,Juventus2018 ~ 2022,20801,188.0,83.0,Right,92,ST,0,"Jul 10, 2018",N/A,46000000,220000,75900000,437,84,95,90,82,86,414,88,81,76,77,92,431,87,91,87,95,71,444,94,95,84,78,93,353,63,29,95,82,84,95,84,28,32,24,58,7,11,15,14,11,2221,464,4,5,High,Low,5,89,93,81,89,35,77,344,Juventus,2018 - 2022
https://cdn.sofifa.com/players/200/389/21_60.png,Jan Oblak,http://sofifa.com/player/200389/jan-oblak/210005/,Slovenia,GK,J. Oblak,27,91,93,Atlético Madrid2014 ~ 2023,200389,188.0,87.1,Right,91,GK,2,"Jul 16, 2014",N/A,75000000,125000,159400000,95,13,11,15,43,13,109,12,13,14,40,30,307,43,60,67,88,49,268,59,78,41,78,12,140,34,19,11,65,11,68,57,27,12,18,437,87,92,78,90,90,1413,489,3,1,Medium,Medium,3,87,92,78,90,52,90,86,Atlético Madrid,2014 - 2023
https://cdn.sofifa.com/players/192/985/21_60.png,Kevin De Bruyne,http://sofifa.com/player/192985/kevin-de-bruyne/210005/,Belgium,CAM CM,K. De Bruyne,29,91,91,Manchester City2015 ~ 2023,192985,180.3,69.9,Right,91,CAM,0,"Aug 30, 2015",N/A,87000000,370000,161000000,407,94,82,55,94,82,441,88,85,83,93,92,398,77,76,78,91,76,408,91,63,89,74,91,408,76,66,88,94,84,91,186,68,65,53,56,15,13,5,10,13,2304,485,5,4,High,High,4,76,86,93,88,64,78,163,Manchester City,2015 - 2023
https://cdn.sofifa.com/players/190/871/21_60.png,Neymar da Silva Santos Jr.,http://sofifa.com/player/190871/neymar-da-silva-santos-jr/210005/,Brazil,LW CAM,Neymar Jr,28,91,91,Paris Saint-Germain2017 ~ 2022,190871,175.3,68.0,Right,91,LW,0,"Aug 3, 2017",N/A,90000000,270000,166500000,408,85,87,62,87,87,448,95,88,89,81,95,453,94,89,96,91,83,357,80,62,81,50,84,356,51,36,87,90,92,93,94,35,30,29,59,9,9,15,15,11,2175,451,5,5,High,Medium,5,91,85,86,94,36,59,273,Paris Saint-Germain,2017 - 2022
https://cdn.sofifa.com/players/188/545/21_60.png,Robert Lewandowski,http://sofifa.com/player/188545/robert-lewandowski/210005/,Poland,ST,R. Lewandowski,31,91,91,FC Bayern München2014 ~ 2023,188545,182.9,79.8,Right,91,ST,0,"Jul 1, 2014",N/A,80000000,240000,132000000,423,71,94,85,84,89,407,85,79,85,70,88,407,77,78,77,93,82,420,89,84,76,86,85,391,81,49,94,79,88,88,96,35,42,19,51,15,6,12,8,10,2195,457,4,4,High,Medium,4,78,91,78,85,43,82,182,FC Bayern München,2014 - 2023
https://cdn.sofifa.com/players/231/747/21_60.png,Kylian Mbappé,http://sofifa.com/player/231747/kylian-mbappe/210005/,France,ST LW RW,K. Mbappé,21,90,95,Paris Saint-Germain2018 ~ 2022,231747,177.8,73.0,Right,91,ST,5,"Jul 1, 2018",N/A,105500000,160000,203100000,408,78,91,73,83,83,394,92,79,63,70,90,458,96,96,92,92,82,404,86,77,86,76,79,341,62,38,91,80,70,84,100,34,34,32,42,13,5,7,11,6,2147,466,4,5,High,

In [0]:
from pyspark.sql import functions as f

# 1) team_original limpio (quita '>' y espacios)
df_silver = df_silver.withColumn(
    "team_original",
    f.trim(f.regexp_replace(f.col("team"), r"^\s*>\s*", ""))
)

df_silver = df_silver.withColumn(
    "is_free",
    f.col("team_original").rlike(r"Free$") | f.lower(f.col("team_original")).rlike(r"\bfree\b")
)

df_silver = df_silver.withColumn("team_clean", f.regexp_replace("team_original", r"Free$", ""))

has_year = f.col("team_clean").rlike(r"(19\d{2}|20\d{2})")

df_silver = df_silver.withColumn(
    "team_name",
    f.when(
        has_year,
        f.trim(f.regexp_extract("team_clean", r"^(.*?)(?:19\d{2}|20\d{2}).*$", 1))
    ).otherwise(f.trim(f.col("team_clean")))
)

# FIX ON LOAN: quitar "Jun 30," etc del team_name
df_silver = df_silver.withColumn(
    "team_name",
    f.trim(
        f.regexp_replace(
            f.col("team_name"),
            r"(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s.*$",
            ""
        )
    )
)

df_silver = df_silver.withColumn(
    "contract_raw",
    f.when(
        has_year,
        f.trim(f.regexp_extract("team_clean", r"((?:19\d{2}|20\d{2}).*)", 1))
    ).otherwise(f.lit(None).cast("string"))
)

df_silver = df_silver.withColumn("contract_raw", f.regexp_replace(f.col("contract_raw"), "~", "-"))

# 2) Limpia "On Loan" al final (2021 On Loan -> 2021)
df_silver = df_silver.withColumn(
    "contract_raw",
    f.trim(f.regexp_replace(f.col("contract_raw"), r"(?i)\s*on\s*loan\s*$", ""))
)

df_silver = df_silver.withColumn(
    "contract_raw",
    f.when(f.length(f.col("contract_raw")) == 0, f.lit(None)).otherwise(f.col("contract_raw"))
)

start_year = f.regexp_extract(df_silver["contract_raw"], r"(19\d{2}|20\d{2})", 1)
end_year   = f.regexp_extract(df_silver["contract_raw"], r".*(19\d{2}|20\d{2})", 1)

df_silver = df_silver.withColumn(
    "contract",
    f.when(
        (start_year != "") & (end_year != "") & (start_year != end_year),
        f.concat_ws(" - ", start_year, end_year)
    ).when(
        (end_year != ""),
        end_year
    ).when(
        f.col("is_free"),
        f.lit("Free")
    ).otherwise(f.lit(None).cast("string"))
)

df_silver = df_silver.drop("team_clean")

# 3) Elimina selecciones (Brazil/Uruguay/Ecuador...) dejando solo contratos o Free
df_silver = df_silver.filter(f.col("contract").isNotNull() | f.col("is_free"))

display(df_silver.limit(20))


photourl,longname,playerurl,nationality,positions,name,age,overall_rating,potential,team,id,height,weight,foot,best_overall,best_position,growth,joined,loan_date_end,value,wage,release_clause,attacking,crossing,finishing,heading_accuracy,short_passing,volleys,skill,dribbling,curve,fk_accuracy,long_passing,ball_control,movement,acceleration,sprint_speed,agility,reactions,balance,power,shot_power,jumping,stamina,strength,long_shots,mentality,aggression,interceptions,positioning,vision,penalties,composure,defending,marking,standing_tackle,sliding_tackle,goalkeeping,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes,total_stats,base_stats,weak_foot_starss,skills_stars,atack_contribution,defense_contribution,international_reputation,pac,sho,pas,dri,def,phy,hits,team_name,contract,team_original,is_free,contract_raw
https://cdn.sofifa.com/players/158/023/21_60.png,Lionel Messi,http://sofifa.com/player/158023/lionel-messi/210005/,Argentina,RW ST CF,L. Messi,33,93,93,FC Barcelona2004 ~ 2021,158023,170.2,72.1,Left,93,RW,0,"Jul 1, 2004",N/A,67500000,560000,138400000,429,85,95,70,91,88,470,96,93,94,91,96,451,91,80,91,94,95,389,86,68,72,69,94,347,44,40,93,95,75,96,91,32,35,24,54,6,11,15,14,8,2231,466,4,4,Medium,Low,5,85,92,91,95,38,65,372,FC Barcelona,2004 - 2021,FC Barcelona2004 ~ 2021,false,2004 - 2021
https://cdn.sofifa.com/players/020/801/21_60.png,C. Ronaldo dos Santos Aveiro,http://sofifa.com/player/20801/c-ronaldo-dos-santos-aveiro/210005/,Portugal,ST LW,Cristiano Ronaldo,35,92,92,Juventus2018 ~ 2022,20801,188.0,83.0,Right,92,ST,0,"Jul 10, 2018",N/A,46000000,220000,75900000,437,84,95,90,82,86,414,88,81,76,77,92,431,87,91,87,95,71,444,94,95,84,78,93,353,63,29,95,82,84,95,84,28,32,24,58,7,11,15,14,11,2221,464,4,5,High,Low,5,89,93,81,89,35,77,344,Juventus,2018 - 2022,Juventus2018 ~ 2022,false,2018 - 2022
https://cdn.sofifa.com/players/200/389/21_60.png,Jan Oblak,http://sofifa.com/player/200389/jan-oblak/210005/,Slovenia,GK,J. Oblak,27,91,93,Atlético Madrid2014 ~ 2023,200389,188.0,87.1,Right,91,GK,2,"Jul 16, 2014",N/A,75000000,125000,159400000,95,13,11,15,43,13,109,12,13,14,40,30,307,43,60,67,88,49,268,59,78,41,78,12,140,34,19,11,65,11,68,57,27,12,18,437,87,92,78,90,90,1413,489,3,1,Medium,Medium,3,87,92,78,90,52,90,86,Atlético Madrid,2014 - 2023,Atlético Madrid2014 ~ 2023,false,2014 - 2023
https://cdn.sofifa.com/players/192/985/21_60.png,Kevin De Bruyne,http://sofifa.com/player/192985/kevin-de-bruyne/210005/,Belgium,CAM CM,K. De Bruyne,29,91,91,Manchester City2015 ~ 2023,192985,180.3,69.9,Right,91,CAM,0,"Aug 30, 2015",N/A,87000000,370000,161000000,407,94,82,55,94,82,441,88,85,83,93,92,398,77,76,78,91,76,408,91,63,89,74,91,408,76,66,88,94,84,91,186,68,65,53,56,15,13,5,10,13,2304,485,5,4,High,High,4,76,86,93,88,64,78,163,Manchester City,2015 - 2023,Manchester City2015 ~ 2023,false,2015 - 2023
https://cdn.sofifa.com/players/190/871/21_60.png,Neymar da Silva Santos Jr.,http://sofifa.com/player/190871/neymar-da-silva-santos-jr/210005/,Brazil,LW CAM,Neymar Jr,28,91,91,Paris Saint-Germain2017 ~ 2022,190871,175.3,68.0,Right,91,LW,0,"Aug 3, 2017",N/A,90000000,270000,166500000,408,85,87,62,87,87,448,95,88,89,81,95,453,94,89,96,91,83,357,80,62,81,50,84,356,51,36,87,90,92,93,94,35,30,29,59,9,9,15,15,11,2175,451,5,5,High,Medium,5,91,85,86,94,36,59,273,Paris Saint-Germain,2017 - 2022,Paris Saint-Germain2017 ~ 2022,false,2017 - 2022
https://cdn.sofifa.com/players/188/545/21_60.png,Robert Lewandowski,http://sofifa.com/player/188545/robert-lewandowski/210005/,Poland,ST,R. Lewandowski,31,91,91,FC Bayern München2014 ~ 2023,188545,182.9,79.8,Right,91,ST,0,"Jul 1, 2014",N/A,80000000,240000,132000000,423,71,94,85,84,89,407,85,79,85,70,88,407,77,78,77,93,82,420,89,84,76,86,85,391,81,49,94,79,88,88,96,35,42,19,51,15,6,12,8,10,2195,457,4,4,High,Medium,4,78,91,78,85,43,82,182,FC Bayern München,2014 - 2023,FC Bayern München2014 ~ 2023,false,2014 - 2023
https://cdn.sofifa.com/players/231/747/21_60.png,Kylian Mbappé,http://sofifa.com/player/231747/kylian-m

In [0]:
df_silver = df_silver.withColumn("positions_list", split("positions", " "))

# Creamos la columna con número de posiciones diferentes en las que puede jugar el jugador
df_silver = df_silver.withColumn("num_positions", size("positions_list"))

In [0]:
#En cuanto a los nulos:

#(hemos vuelto a hacer la importacion de funciones porque por algún motivo si la hacemos arriba no funciona esta celda y da error la de suma, además da error más abajo también si no le ponemos el alias porque interfiere con la funcion de sum normal)
from pyspark.sql.functions import col, when, sum as spark_sum

df_silver.select(
    spark_sum(when(col("loan_date_end") == "N/A", 1).otherwise(0)).alias("na_count"),
    spark_sum(when(col("loan_date_end") != "N/A", 1).otherwise(0)).alias("valid_count")
).show()

#Nos quitamos la columna de load_date_end porque no hay casi ningun valor

+--------+-----------+
|na_count|valid_count|
+--------+-----------+
|   17966|       1013|
+--------+-----------+



In [0]:
#nos quitamos estas columnas porque son redundantes y o las hemos usado para sacar otros campos mejores
df_silver = df_silver.drop("photourl", "name", "team","positions_list", "positions","loan_date_end")
display(df_silver.limit(20))

longname,playerurl,nationality,age,overall_rating,potential,id,height,weight,foot,best_overall,best_position,growth,joined,value,wage,release_clause,attacking,crossing,finishing,heading_accuracy,short_passing,volleys,skill,dribbling,curve,fk_accuracy,long_passing,ball_control,movement,acceleration,sprint_speed,agility,reactions,balance,power,shot_power,jumping,stamina,strength,long_shots,mentality,aggression,interceptions,positioning,vision,penalties,composure,defending,marking,standing_tackle,sliding_tackle,goalkeeping,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes,total_stats,base_stats,weak_foot_starss,skills_stars,atack_contribution,defense_contribution,international_reputation,pac,sho,pas,dri,def,phy,hits,team_name,contract,team_original,is_free,contract_raw,num_positions
Lionel Messi,http://sofifa.com/player/158023/lionel-messi/210005/,Argentina,33,93,93,158023,170.2,72.1,Left,93,RW,0,"Jul 1, 2004",67500000,560000,138400000,429,85,95,70,91,88,470,96,93,94,91,96,451,91,80,91,94,95,389,86,68,72,69,94,347,44,40,93,95,75,96,91,32,35,24,54,6,11,15,14,8,2231,466,4,4,Medium,Low,5,85,92,91,95,38,65,372,FC Barcelona,2004 - 2021,FC Barcelona2004 ~ 2021,false,2004 - 2021,3
C. Ronaldo dos Santos Aveiro,http://sofifa.com/player/20801/c-ronaldo-dos-santos-aveiro/210005/,Portugal,35,92,92,20801,188.0,83.0,Right,92,ST,0,"Jul 10, 2018",46000000,220000,75900000,437,84,95,90,82,86,414,88,81,76,77,92,431,87,91,87,95,71,444,94,95,84,78,93,353,63,29,95,82,84,95,84,28,32,24,58,7,11,15,14,11,2221,464,4,5,High,Low,5,89,93,81,89,35,77,344,Juventus,2018 - 2022,Juventus2018 ~ 2022,false,2018 - 2022,2
Jan Oblak,http://sofifa.com/player/200389/jan-oblak/210005/,Slovenia,27,91,93,200389,188.0,87.1,Right,91,GK,2,"Jul 16, 2014",75000000,125000,159400000,95,13,11,15,43,13,109,12,13,14,40,30,307,43,60,67,88,49,268,59,78,41,78,12,140,34,19,11,65,11,68,57,27,12,18,437,87,92,78,90,90,1413,489,3,1,Medium,Medium,3,87,92,78,90,52,90,86,Atlético Madrid,2014 - 2023,Atlético Madrid2014 ~ 2023,false,2014 - 2023,1
Kevin De Bruyne,http://sofifa.com/player/192985/kevin-de-bruyne/210005/,Belgium,29,91,91,192985,180.3,69.9,Right,91,CAM,0,"Aug 30, 2015",87000000,370000,161000000,407,94,82,55,94,82,441,88,85,83,93,92,398,77,76,78,91,76,408,91,63,89,74,91,408,76,66,88,94,84,91,186,68,65,53,56,15,13,5,10,13,2304,485,5,4,High,High,4,76,86,93,88,64,78,163,Manchester City,2015 - 2023,Manchester City2015 ~ 2023,false,2015 - 2023,2
Neymar da Silva Santos Jr.,http://sofifa.com/player/190871/neymar-da-silva-santos-jr/210005/,Brazil,28,91,91,190871,175.3,68.0,Right,91,LW,0,"Aug 3, 2017",90000000,270000,166500000,408,85,87,62,87,87,448,95,88,89,81,95,453,94,89,96,91,83,357,80,62,81,50,84,356,51,36,87,90,92,93,94,35,30,29,59,9,9,15,15,11,2175,451,5,5,High,Medium,5,91,85,86,94,36,59,273,Paris Saint-Germain,2017 - 2022,Paris Saint-Germain2017 ~ 2022,false,2017 - 2022,2
Robert Lewandowski,http://sofifa.com/player/188545/robert-lewandowski/210005/,Poland,31,91,91,188545,182.9,79.8,Right,91,ST,0,"Jul 1, 2014",80000000,240000,132000000,423,71,94,85,84,89,407,85,79,85,70,88,407,77,78,77,93,82,420,89,84,76,86,85,391,81,49,94,79,88,88,96,35,42,19,51,15,6,12,8,10,2195,457,4,4,High,Medium,4,78,91,78,85,43,82,182,FC Bayern München,2014 - 2023,FC Bayern München2014 ~ 2023,false,2014 - 2023,1
Kylian Mbappé,http://sofifa.com/player/231747/kylian-mbappe/210005/,France,21,90,95,231747,177.8,73.0,Right,91,ST,5,"Jul 1, 2018",105500000,160000,203100000,408,78,91,73,83,83,394,92,79,63,70,90,458,96,96,92,92,82,404,86,77,86,76,79,341,62,38,91,80,70,84,100,34,34,32,42,13,5,7,11,6,2147,466,4,5,High,Low,3,96,86,78,91,39,76,646,Paris Saint-Germain,2018 - 2022,Paris Saint-Germain2018 ~ 2022,false,2018 - 2022,3
Alisson Ramses Becker,http://sofifa.com/player/212831/alisson-ramses-becker/210005/,Brazil,27,90,91,212831,190.5,91.2,Right,90,GK,1,"Jul 19, 2018",62500000,160000,120300000,114,17,13,19,45,20,138,27,19,18,44,30,268,56,47,40,88,37,240,64,52,32,78,14,140,27,11,13,66,23,65,50,15,19,16,439,86,88,

In [0]:
#Aquí podemos ver que hay jugadores con mismo nombre, pero que son jugadores diferentes, por lo que no son necesariamente duplicados.
df_silver.groupBy("longname").agg(f.count("*").alias("count")).filter(col("count") > 1).show()

+------------------+-----+
|          longname|count|
+------------------+-----+
|   Alejandro Gómez|    2|
|      Stefan Savić|    2|
|        Ben Davies|    2|
|        Danny Rose|    3|
|         Luis Díaz|    2|
|Jonathan Rodríguez|    2|
| Emiliano Martínez|    2|
|  Patrick Herrmann|    2|
| Ricardo Rodríguez|    2|
|    Lisandro López|    2|
|     Kevin Berlaso|    2|
|     Claudio Bravo|    2|
|       Reece James|    2|
|  Emmanuel Boateng|    2|
|    Gonzalo Castro|    2|
|   Santiago García|    2|
|  Cristian Álvarez|    2|
|     Jorge Sánchez|    2|
|  Nicolás González|    3|
|        Tom Davies|    2|
+------------------+-----+
only showing top 20 rows


In [0]:
df_filtered = df_silver.filter(col("longname").contains("Bravo"))

display(df_filtered.limit(20))

#Puede haber varios jugadores con mismo nombre

longname,playerurl,nationality,age,overall_rating,potential,id,height,weight,foot,best_overall,best_position,growth,joined,value,wage,release_clause,attacking,crossing,finishing,heading_accuracy,short_passing,volleys,skill,dribbling,curve,fk_accuracy,long_passing,ball_control,movement,acceleration,sprint_speed,agility,reactions,balance,power,shot_power,jumping,stamina,strength,long_shots,mentality,aggression,interceptions,positioning,vision,penalties,composure,defending,marking,standing_tackle,sliding_tackle,goalkeeping,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes,total_stats,base_stats,weak_foot_starss,skills_stars,atack_contribution,defense_contribution,international_reputation,pac,sho,pas,dri,def,phy,hits,team_name,contract,team_original,is_free,contract_raw,num_positions
Claudio Bravo,http://sofifa.com/player/174543/claudio-bravo/210005/,Chile,37,77,77,174543,182.9,79.8,Right,77,GK,0,"Aug 30, 2020",900000,12000,1900000,110,12,13,18,56,11,171,22,25,38,54,32,309,58,54,63,72,62,269,63,81,39,69,17,171,40,23,16,69,23,64,52,15,18,19,389,77,76,84,75,77,1471,445,3,1,Medium,Medium,3,77,76,84,77,56,75,34,Real Betis,2020 - 2022,Real Betis2020 ~ 2022,false,2020 - 2022,1
Claudio Bravo,http://sofifa.com/player/232646/claudio-bravo/210005/,Argentina,23,73,82,232646,170.2,68.9,Left,74,LWB,9,"Jan 4, 2016",5500000,10000,9400000,282,70,48,53,66,45,296,76,51,38,60,71,359,77,72,72,63,75,308,59,71,74,57,47,321,73,75,57,69,47,64,218,71,73,74,58,14,12,8,11,13,1842,397,2,3,Medium,Medium,1,74,50,64,73,71,65,40,Club Atlético Banfield,2016 - 2021,Club Atlético Banfield2016 ~ 2021,false,2016 - 2021,1
Christian Bravo,http://sofifa.com/player/219104/christian-bravo/210005/,Chile,26,67,67,219104,167.6,64.0,Right,67,RW,0,"Jan 19, 2020",825000,500,1800000,291,62,55,51,57,66,297,71,52,53,54,67,421,89,90,89,62,91,358,75,87,61,77,58,245,43,23,67,52,60,55,89,29,33,27,58,12,14,7,16,9,1759,377,3,3,High,Medium,1,90,61,56,72,31,67,3,Peñarol,2020,Peñarol2020 ~ 2020,false,2020 - 2020,3
Iván Bravo Castro,http://sofifa.com/player/258768/ivan-bravo-castro/210005/,Spain,19,65,77,258768,177.8,64.0,Left,65,LB,12,"Jun 1, 2019",850000,1000,1500000,222,54,33,52,54,29,257,65,49,39,50,54,345,78,76,60,58,73,244,29,55,75,50,35,234,55,59,40,36,44,45,190,58,67,65,59,14,8,13,15,9,1551,339,3,3,High,Medium,1,77,33,49,61,61,58,4,RCD Mallorca,2019 - 2021,RCD Mallorca2019 ~ 2021,false,2019 - 2021,1
Juan Manuel Bravo Alcántara,http://sofifa.com/player/258557/juan-manuel-bravo-alcantara/210005/,Spain,22,64,73,258557,188.0,73.9,Right,66,CM,9,"Sep 1, 2020",675000,2000,1200000,291,54,55,66,70,46,291,63,48,47,67,66,321,66,63,68,63,61,309,70,54,59,67,59,307,70,59,58,60,60,56,180,61,62,57,54,7,13,12,14,8,1753,375,3,2,Medium,Medium,1,64,59,62,64,61,65,3,AD Alcorcón,2020 - 2024,AD Alcorcón2020 ~ 2024,false,2020 - 2024,1
Endika Irigoyen Bravo,http://sofifa.com/player/244005/endika-irigoyen-bravo/210005/,Spain,23,63,70,244005,177.8,68.0,Left,63,LB,7,"Jul 1, 2015",450000,5000,1000000,241,57,35,59,56,34,240,55,46,41,44,54,317,68,64,54,63,68,268,35,56,71,69,37,254,68,60,42,43,41,44,179,55,61,63,44,13,9,9,5,8,1543,337,2,2,Medium,High,1,66,36,51,56,59,69,2,CA Osasuna,2015 - 2022,CA Osasuna2015 ~ 2022,false,2015 - 2022,1
Eduardo Bravo,http://sofifa.com/player/257385/eduardo-bravo/210005/,Mexico,29,57,57,257385,182.9,82.1,Right,57,GK,0,"Jul 1, 2020",70000,2000,116000,59,14,9,12,16,8,67,11,10,12,20,14,168,31,25,21,45,46,178,45,65,21,40,7,75,26,8,6,21,14,36,29,8,10,11,291,58,55,60,58,60,867,319,3,1,Medium,Medium,1,58,55,60,60,28,58,3,Querétaro,2020 - 2024,Querétaro2020 ~ 2024,false,2020 - 2024,1


In [0]:
#Sin embargo si que deberían tener id suficiente

df_silver.groupBy("id").agg(f.count("*").alias("count")).filter(col("count") > 1).show()

+------+-----+
|    id|count|
+------+-----+
|251698|    2|
+------+-----+



In [0]:
df_filtered = df_silver.filter(col("id").contains("251698"))

display(df_filtered.limit(20))

#Este SI es un duplicado, hay que eliminar por lo tanto duplicados en la columna ID

longname,playerurl,nationality,age,overall_rating,potential,id,height,weight,foot,best_overall,best_position,growth,joined,value,wage,release_clause,attacking,crossing,finishing,heading_accuracy,short_passing,volleys,skill,dribbling,curve,fk_accuracy,long_passing,ball_control,movement,acceleration,sprint_speed,agility,reactions,balance,power,shot_power,jumping,stamina,strength,long_shots,mentality,aggression,interceptions,positioning,vision,penalties,composure,defending,marking,standing_tackle,sliding_tackle,goalkeeping,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes,total_stats,base_stats,weak_foot_starss,skills_stars,atack_contribution,defense_contribution,international_reputation,pac,sho,pas,dri,def,phy,hits,team_name,contract,team_original,is_free,contract_raw,num_positions
Kevin Berlaso,http://sofifa.com/player/251698/kevin-berlaso/210005/,Ecuador,32,77,77,251698,172.7,68.9,Right,77,RB,0,"Jan 1, 2010",0,0,0,306,72,47,60,73,54,350,75,75,54,68,78,397,77,78,86,77,79,345,69,70,86,57,63,323,73,70,69,63,48,73,224,71,75,78,58,11,12,11,16,8,2003,420,3,4,High,Medium,2,78,56,69,77,72,68,12,Ecuador,Free,EcuadorFree,true,null,1
Kevin Berlaso,http://sofifa.com/player/251698/kevin-berlaso/210005/,Ecuador,32,77,77,251698,172.7,68.9,Right,77,RB,0,"Jan 1, 2010",0,0,0,306,72,47,60,73,54,350,75,75,54,68,78,397,77,78,86,77,79,345,69,70,86,57,63,323,73,70,69,63,48,73,224,71,75,78,58,11,12,11,16,8,2003,420,3,4,High,Medium,2,78,56,69,77,72,68,12,Ecuador,Free,EcuadorFree,true,null,1


In [0]:
df_silver = df_silver.dropDuplicates(["id"])

In [0]:
display(df_silver.limit(20))

longname,playerurl,nationality,age,overall_rating,potential,id,height,weight,foot,best_overall,best_position,growth,joined,value,wage,release_clause,attacking,crossing,finishing,heading_accuracy,short_passing,volleys,skill,dribbling,curve,fk_accuracy,long_passing,ball_control,movement,acceleration,sprint_speed,agility,reactions,balance,power,shot_power,jumping,stamina,strength,long_shots,mentality,aggression,interceptions,positioning,vision,penalties,composure,defending,marking,standing_tackle,sliding_tackle,goalkeeping,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes,total_stats,base_stats,weak_foot_starss,skills_stars,atack_contribution,defense_contribution,international_reputation,pac,sho,pas,dri,def,phy,hits,team_name,contract,team_original,is_free,contract_raw,num_positions
Lionel Messi,http://sofifa.com/player/158023/lionel-messi/210005/,Argentina,33,93,93,158023,170.2,72.1,Left,93,RW,0,"Jul 1, 2004",67500000,560000,138400000,429,85,95,70,91,88,470,96,93,94,91,96,451,91,80,91,94,95,389,86,68,72,69,94,347,44,40,93,95,75,96,91,32,35,24,54,6,11,15,14,8,2231,466,4,4,Medium,Low,5,85,92,91,95,38,65,372,FC Barcelona,2004 - 2021,FC Barcelona2004 ~ 2021,false,2004 - 2021,3
C. Ronaldo dos Santos Aveiro,http://sofifa.com/player/20801/c-ronaldo-dos-santos-aveiro/210005/,Portugal,35,92,92,20801,188.0,83.0,Right,92,ST,0,"Jul 10, 2018",46000000,220000,75900000,437,84,95,90,82,86,414,88,81,76,77,92,431,87,91,87,95,71,444,94,95,84,78,93,353,63,29,95,82,84,95,84,28,32,24,58,7,11,15,14,11,2221,464,4,5,High,Low,5,89,93,81,89,35,77,344,Juventus,2018 - 2022,Juventus2018 ~ 2022,false,2018 - 2022,2
Jan Oblak,http://sofifa.com/player/200389/jan-oblak/210005/,Slovenia,27,91,93,200389,188.0,87.1,Right,91,GK,2,"Jul 16, 2014",75000000,125000,159400000,95,13,11,15,43,13,109,12,13,14,40,30,307,43,60,67,88,49,268,59,78,41,78,12,140,34,19,11,65,11,68,57,27,12,18,437,87,92,78,90,90,1413,489,3,1,Medium,Medium,3,87,92,78,90,52,90,86,Atlético Madrid,2014 - 2023,Atlético Madrid2014 ~ 2023,false,2014 - 2023,1
Kevin De Bruyne,http://sofifa.com/player/192985/kevin-de-bruyne/210005/,Belgium,29,91,91,192985,180.3,69.9,Right,91,CAM,0,"Aug 30, 2015",87000000,370000,161000000,407,94,82,55,94,82,441,88,85,83,93,92,398,77,76,78,91,76,408,91,63,89,74,91,408,76,66,88,94,84,91,186,68,65,53,56,15,13,5,10,13,2304,485,5,4,High,High,4,76,86,93,88,64,78,163,Manchester City,2015 - 2023,Manchester City2015 ~ 2023,false,2015 - 2023,2
Neymar da Silva Santos Jr.,http://sofifa.com/player/190871/neymar-da-silva-santos-jr/210005/,Brazil,28,91,91,190871,175.3,68.0,Right,91,LW,0,"Aug 3, 2017",90000000,270000,166500000,408,85,87,62,87,87,448,95,88,89,81,95,453,94,89,96,91,83,357,80,62,81,50,84,356,51,36,87,90,92,93,94,35,30,29,59,9,9,15,15,11,2175,451,5,5,High,Medium,5,91,85,86,94,36,59,273,Paris Saint-Germain,2017 - 2022,Paris Saint-Germain2017 ~ 2022,false,2017 - 2022,2
Robert Lewandowski,http://sofifa.com/player/188545/robert-lewandowski/210005/,Poland,31,91,91,188545,182.9,79.8,Right,91,ST,0,"Jul 1, 2014",80000000,240000,132000000,423,71,94,85,84,89,407,85,79,85,70,88,407,77,78,77,93,82,420,89,84,76,86,85,391,81,49,94,79,88,88,96,35,42,19,51,15,6,12,8,10,2195,457,4,4,High,Medium,4,78,91,78,85,43,82,182,FC Bayern München,2014 - 2023,FC Bayern München2014 ~ 2023,false,2014 - 2023,1
Kylian Mbappé,http://sofifa.com/player/231747/kylian-mbappe/210005/,France,21,90,95,231747,177.8,73.0,Right,91,ST,5,"Jul 1, 2018",105500000,160000,203100000,408,78,91,73,83,83,394,92,79,63,70,90,458,96,96,92,92,82,404,86,77,86,76,79,341,62,38,91,80,70,84,100,34,34,32,42,13,5,7,11,6,2147,466,4,5,High,Low,3,96,86,78,91,39,76,646,Paris Saint-Germain,2018 - 2022,Paris Saint-Germain2018 ~ 2022,false,2018 - 2022,3
Alisson Ramses Becker,http://sofifa.com/player/212831/alisson-ramses-becker/210005/,Brazil,27,90,91,212831,190.5,91.2,Right,90,GK,1,"Jul 19, 2018",62500000,160000,120300000,114,17,13,19,45,20,138,27,19,18,44,30,268,56,47,40,88,37,240,64,52,32,78,14,140,27,11,13,66,23,65,50,15,19,16,439,86,88,

In [0]:
df_silver.columns

['longname',
 'playerurl',
 'nationality',
 'age',
 'overall_rating',
 'potential',
 'id',
 'height',
 'weight',
 'foot',
 'best_overall',
 'best_position',
 'growth',
 'joined',
 'value',
 'wage',
 'release_clause',
 'attacking',
 'crossing',
 'finishing',
 'heading_accuracy',
 'short_passing',
 'volleys',
 'skill',
 'dribbling',
 'curve',
 'fk_accuracy',
 'long_passing',
 'ball_control',
 'movement',
 'acceleration',
 'sprint_speed',
 'agility',
 'reactions',
 'balance',
 'power',
 'shot_power',
 'jumping',
 'stamina',
 'strength',
 'long_shots',
 'mentality',
 'aggression',
 'interceptions',
 'positioning',
 'vision',
 'penalties',
 'composure',
 'defending',
 'marking',
 'standing_tackle',
 'sliding_tackle',
 'goalkeeping',
 'gk_diving',
 'gk_handling',
 'gk_kicking',
 'gk_positioning',
 'gk_reflexes',
 'total_stats',
 'base_stats',
 'weak_foot_starss',
 'skills_stars',
 'atack_contribution',
 'defense_contribution',
 'international_reputation',
 'pac',
 'sho',
 'pas',
 'dri',
 'de

Ahora sacaremos el año de inicio del contrato, luego calcularemmos cuántos años lleva en el equipo tomando 2021 como referencia

In [0]:
# Sacamos el año de inicio del contrato 
df_silver = df_silver.withColumn(
    "contract_start_year",
    f.expr("try_cast(regexp_extract(contract, '(\\\\d{4})', 1) as int)")
)

# Años en el equipo tomando 2021 como referencia
df_silver = df_silver.withColumn(
    "years_at_team_2021",
    f.when(f.col("contract_start_year").isNotNull(),
           f.lit(2021) - f.col("contract_start_year")
    ).otherwise(None)
)

df_silver.select("team_name", "contract", "contract_start_year", "years_at_team_2021").show(20, False)


+-------------------+-----------+-------------------+------------------+
|team_name          |contract   |contract_start_year|years_at_team_2021|
+-------------------+-----------+-------------------+------------------+
|FC Barcelona       |2004 - 2021|2004               |17                |
|Juventus           |2018 - 2022|2018               |3                 |
|Atlético Madrid    |2014 - 2023|2014               |7                 |
|Manchester City    |2015 - 2023|2015               |6                 |
|Paris Saint-Germain|2017 - 2022|2017               |4                 |
|FC Bayern München  |2014 - 2023|2014               |7                 |
|Paris Saint-Germain|2018 - 2022|2018               |3                 |
|Liverpool          |2018 - 2024|2018               |3                 |
|Liverpool          |2017 - 2023|2017               |4                 |
|Liverpool          |2016 - 2023|2016               |5                 |
|Liverpool          |2018 - 2023|2018              

In [0]:
display(df_silver.limit(20))

longname,playerurl,nationality,age,overall_rating,potential,id,height,weight,foot,best_overall,best_position,growth,joined,value,wage,release_clause,attacking,crossing,finishing,heading_accuracy,short_passing,volleys,skill,dribbling,curve,fk_accuracy,long_passing,ball_control,movement,acceleration,sprint_speed,agility,reactions,balance,power,shot_power,jumping,stamina,strength,long_shots,mentality,aggression,interceptions,positioning,vision,penalties,composure,defending,marking,standing_tackle,sliding_tackle,goalkeeping,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes,total_stats,base_stats,weak_foot_starss,skills_stars,atack_contribution,defense_contribution,international_reputation,pac,sho,pas,dri,def,phy,hits,team_name,contract,team_original,is_free,contract_raw,num_positions,contract_start_year,years_at_team_2021
Lionel Messi,http://sofifa.com/player/158023/lionel-messi/210005/,Argentina,33,93,93,158023,170.2,72.1,Left,93,RW,0,"Jul 1, 2004",67500000,560000,138400000,429,85,95,70,91,88,470,96,93,94,91,96,451,91,80,91,94,95,389,86,68,72,69,94,347,44,40,93,95,75,96,91,32,35,24,54,6,11,15,14,8,2231,466,4,4,Medium,Low,5,85,92,91,95,38,65,372,FC Barcelona,2004 - 2021,FC Barcelona2004 ~ 2021,false,2004 - 2021,3,2004,17
C. Ronaldo dos Santos Aveiro,http://sofifa.com/player/20801/c-ronaldo-dos-santos-aveiro/210005/,Portugal,35,92,92,20801,188.0,83.0,Right,92,ST,0,"Jul 10, 2018",46000000,220000,75900000,437,84,95,90,82,86,414,88,81,76,77,92,431,87,91,87,95,71,444,94,95,84,78,93,353,63,29,95,82,84,95,84,28,32,24,58,7,11,15,14,11,2221,464,4,5,High,Low,5,89,93,81,89,35,77,344,Juventus,2018 - 2022,Juventus2018 ~ 2022,false,2018 - 2022,2,2018,3
Jan Oblak,http://sofifa.com/player/200389/jan-oblak/210005/,Slovenia,27,91,93,200389,188.0,87.1,Right,91,GK,2,"Jul 16, 2014",75000000,125000,159400000,95,13,11,15,43,13,109,12,13,14,40,30,307,43,60,67,88,49,268,59,78,41,78,12,140,34,19,11,65,11,68,57,27,12,18,437,87,92,78,90,90,1413,489,3,1,Medium,Medium,3,87,92,78,90,52,90,86,Atlético Madrid,2014 - 2023,Atlético Madrid2014 ~ 2023,false,2014 - 2023,1,2014,7
Kevin De Bruyne,http://sofifa.com/player/192985/kevin-de-bruyne/210005/,Belgium,29,91,91,192985,180.3,69.9,Right,91,CAM,0,"Aug 30, 2015",87000000,370000,161000000,407,94,82,55,94,82,441,88,85,83,93,92,398,77,76,78,91,76,408,91,63,89,74,91,408,76,66,88,94,84,91,186,68,65,53,56,15,13,5,10,13,2304,485,5,4,High,High,4,76,86,93,88,64,78,163,Manchester City,2015 - 2023,Manchester City2015 ~ 2023,false,2015 - 2023,2,2015,6
Neymar da Silva Santos Jr.,http://sofifa.com/player/190871/neymar-da-silva-santos-jr/210005/,Brazil,28,91,91,190871,175.3,68.0,Right,91,LW,0,"Aug 3, 2017",90000000,270000,166500000,408,85,87,62,87,87,448,95,88,89,81,95,453,94,89,96,91,83,357,80,62,81,50,84,356,51,36,87,90,92,93,94,35,30,29,59,9,9,15,15,11,2175,451,5,5,High,Medium,5,91,85,86,94,36,59,273,Paris Saint-Germain,2017 - 2022,Paris Saint-Germain2017 ~ 2022,false,2017 - 2022,2,2017,4
Robert Lewandowski,http://sofifa.com/player/188545/robert-lewandowski/210005/,Poland,31,91,91,188545,182.9,79.8,Right,91,ST,0,"Jul 1, 2014",80000000,240000,132000000,423,71,94,85,84,89,407,85,79,85,70,88,407,77,78,77,93,82,420,89,84,76,86,85,391,81,49,94,79,88,88,96,35,42,19,51,15,6,12,8,10,2195,457,4,4,High,Medium,4,78,91,78,85,43,82,182,FC Bayern München,2014 - 2023,FC Bayern München2014 ~ 2023,false,2014 - 2023,1,2014,7
Kylian Mbappé,http://sofifa.com/player/231747/kylian-mbappe/210005/,France,21,90,95,231747,177.8,73.0,Right,91,ST,5,"Jul 1, 2018",105500000,160000,203100000,408,78,91,73,83,83,394,92,79,63,70,90,458,96,96,92,92,82,404,86,77,86,76,79,341,62,38,91,80,70,84,100,34,34,32,42,13,5,7,11,6,2147,466,4,5,High,Low,3,96,86,78,91,39,76,646,Paris Saint-Germain,2018 - 2022,Paris Saint-Germain2018 ~ 2022,false,2018 - 2022,3,2018,3
Alisson Ramses Becker,http://sofifa.com/player/212831/alisson-ramses-becker/210005/,Brazil,27,90,91,212831,190.5,91.2,Right,90,GK,1,"Jul 19, 2018",62500000,160000,120300000,114,17,13,19,45,20,138,27,19,18

Ahora vamos a crear una columa categórica, donde dependiendo de lo que haya costado el jugador:
-   Low (menos de 5M)
-   Mid (entre 5M y 20M)
-   High  (entre 20M y 60M)
-   Premium (más de 60M)


Además, crearemos otra columna categorizando por edad: 

-      Young (menos de 22 años)
-      Prime (Entre 22  y 28 años)
-      Veteran (Entre 28 y 34 años)
-      Late career (Más de 34 años)

In [0]:

df_silver = df_silver.withColumn(
    "value_player",
    f.when(f.col("value") < 5_000_000, "Low")
     .when((f.col("value") >= 5_000_000) & (f.col("value") < 20_000_000), "Mid")
     .when((f.col("value") >= 20_000_000) & (f.col("value") < 60_000_000), "High")
     .otherwise("Premium")
)

df_silver = df_silver.withColumn(
    "age_category",
    f.when(f.col("age") < 22, "Young")
     .when(f.col("age") <= 28, "Prime")
     
     .when(f.col("age") <= 34, "Veteran")
     .otherwise("Late")
)



A continuación, vamos a crear otras 2 columnas: "offensive_score" y "defensive score"

En ella calcularemos la media de las métricas de ataque y de defensa de cada jugador

In [0]:
df_silver = df_silver.withColumn(
    "offensive_score",
    f.round(
        (f.col("finishing") + f.col("long_shots") + f.col("dribbling") +
         f.col("curve") + f.col("ball_control") + f.col("acceleration")) / 6,
        2
    )
)

df_silver = df_silver.withColumn(
    "defensive_score",
    f.round(
        (f.col("standing_tackle") + f.col("sliding_tackle") + f.col("interceptions") +
         f.col("def") + f.col("strength")) / 5,
        2
    )
)


In [0]:
df_silver.groupBy("weak_foot_starss") \
    .count() \
    .orderBy("weak_foot_starss") \
    .show()

+----------------+-----+
|weak_foot_starss|count|
+----------------+-----+
|               1|  138|
|               2| 4141|
|               3|11694|
|               4| 2722|
|               5|  283|
+----------------+-----+



Vamos a crear otra nueva columna categórica llamada "two_footed", la cual  indicará si el jugador es bueno con las dos piernas. Utilizaremos como referencia si la columna "weak_foot_starss" tiene un valor mayor  a 4. 

Esta será una columna categórica binaria, con un 1 indicando si el jugador es potencialmente ambidiestro y 0 si no lo es.

In [0]:
df_silver = df_silver.withColumn(
    "two_footed",
    f.when(f.col("weak_foot_starss") >= 4, 1).otherwise(0)
)


Por último, vamos a crear otra columna llamada "clutch_score", donde se reflejará si el jugador es bueno en estos 3 ámbitos: 
- composure (sangre fría)
- vision (toma de decisiones)
- reactions (velocidad de reacción)

In [0]:
df_silver = df_silver.withColumn(
    "clutch_score",
    f.round(
        (f.col("composure") + f.col("vision") + f.col("reactions")) / 3,
        2
    )
)
# Crearemos otra columna  para categorizarlo  en "Elite", "Strong", "Average" o "Weak"

df_silver = df_silver.withColumn(
    "clutch_level",
    f.when(f.col("clutch_score") >= 85, "Elite")
     .when(f.col("clutch_score") >= 70, "Strong")
     .when(f.col("clutch_score") >= 55, "Average")
     .otherwise("Weak")
)

In [0]:
display(df_silver.limit(20))

longname,playerurl,nationality,age,overall_rating,potential,id,height,weight,foot,best_overall,best_position,growth,joined,value,wage,release_clause,attacking,crossing,finishing,heading_accuracy,short_passing,volleys,skill,dribbling,curve,fk_accuracy,long_passing,ball_control,movement,acceleration,sprint_speed,agility,reactions,balance,power,shot_power,jumping,stamina,strength,long_shots,mentality,aggression,interceptions,positioning,vision,penalties,composure,defending,marking,standing_tackle,sliding_tackle,goalkeeping,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes,total_stats,base_stats,weak_foot_starss,skills_stars,atack_contribution,defense_contribution,international_reputation,pac,sho,pas,dri,def,phy,hits,team_name,contract,team_original,is_free,contract_raw,num_positions,contract_start_year,years_at_team_2021,value_player,age_category,offensive_score,defensive_score,two_footed,clutch_score,clutch_level
Lionel Messi,http://sofifa.com/player/158023/lionel-messi/210005/,Argentina,33,93,93,158023,170.2,72.1,Left,93,RW,0,"Jul 1, 2004",67500000,560000,138400000,429,85,95,70,91,88,470,96,93,94,91,96,451,91,80,91,94,95,389,86,68,72,69,94,347,44,40,93,95,75,96,91,32,35,24,54,6,11,15,14,8,2231,466,4,4,Medium,Low,5,85,92,91,95,38,65,372,FC Barcelona,2004 - 2021,FC Barcelona2004 ~ 2021,false,2004 - 2021,3,2004,17,Premium,Veteran,94.17,41.2,1,95.0,Elite
C. Ronaldo dos Santos Aveiro,http://sofifa.com/player/20801/c-ronaldo-dos-santos-aveiro/210005/,Portugal,35,92,92,20801,188.0,83.0,Right,92,ST,0,"Jul 10, 2018",46000000,220000,75900000,437,84,95,90,82,86,414,88,81,76,77,92,431,87,91,87,95,71,444,94,95,84,78,93,353,63,29,95,82,84,95,84,28,32,24,58,7,11,15,14,11,2221,464,4,5,High,Low,5,89,93,81,89,35,77,344,Juventus,2018 - 2022,Juventus2018 ~ 2022,false,2018 - 2022,2,2018,3,High,Late,89.33,39.6,1,90.67,Elite
Jan Oblak,http://sofifa.com/player/200389/jan-oblak/210005/,Slovenia,27,91,93,200389,188.0,87.1,Right,91,GK,2,"Jul 16, 2014",75000000,125000,159400000,95,13,11,15,43,13,109,12,13,14,40,30,307,43,60,67,88,49,268,59,78,41,78,12,140,34,19,11,65,11,68,57,27,12,18,437,87,92,78,90,90,1413,489,3,1,Medium,Medium,3,87,92,78,90,52,90,86,Atlético Madrid,2014 - 2023,Atlético Madrid2014 ~ 2023,false,2014 - 2023,1,2014,7,Premium,Prime,20.17,35.8,0,73.67,Strong
Kevin De Bruyne,http://sofifa.com/player/192985/kevin-de-bruyne/210005/,Belgium,29,91,91,192985,180.3,69.9,Right,91,CAM,0,"Aug 30, 2015",87000000,370000,161000000,407,94,82,55,94,82,441,88,85,83,93,92,398,77,76,78,91,76,408,91,63,89,74,91,408,76,66,88,94,84,91,186,68,65,53,56,15,13,5,10,13,2304,485,5,4,High,High,4,76,86,93,88,64,78,163,Manchester City,2015 - 2023,Manchester City2015 ~ 2023,false,2015 - 2023,2,2015,6,Premium,Veteran,85.83,64.4,1,92.0,Elite
Neymar da Silva Santos Jr.,http://sofifa.com/player/190871/neymar-da-silva-santos-jr/210005/,Brazil,28,91,91,190871,175.3,68.0,Right,91,LW,0,"Aug 3, 2017",90000000,270000,166500000,408,85,87,62,87,87,448,95,88,89,81,95,453,94,89,96,91,83,357,80,62,81,50,84,356,51,36,87,90,92,93,94,35,30,29,59,9,9,15,15,11,2175,451,5,5,High,Medium,5,91,85,86,94,36,59,273,Paris Saint-Germain,2017 - 2022,Paris Saint-Germain2017 ~ 2022,false,2017 - 2022,2,2017,4,Premium,Prime,90.5,36.2,1,91.33,Elite
Robert Lewandowski,http://sofifa.com/player/188545/robert-lewandowski/210005/,Poland,31,91,91,188545,182.9,79.8,Right,91,ST,0,"Jul 1, 2014",80000000,240000,132000000,423,71,94,85,84,89,407,85,79,85,70,88,407,77,78,77,93,82,420,89,84,76,86,85,391,81,49,94,79,88,88,96,35,42,19,51,15,6,12,8,10,2195,457,4,4,High,Medium,4,78,91,78,85,43,82,182,FC Bayern München,2014 - 2023,FC Bayern München2014 ~ 2023,false,2014 - 2023,1,2014,7,Premium,Veteran,84.67,47.8,1,86.67,Elite
Kylian Mbappé,http://sofifa.com/player/231747/kylian-mbappe/210005/,France,21,90,95,231747,177.8,73.0,Right,91,ST,5,"Jul 1, 2018",105500000,160000,203100000,408,78,91,73,83,83,394,92,79,63,70,90,458,96,96,92,92,82,404,86,77,86,76,79,341,62,38,91,80,70,84,100,34,34,32,42,13,5,7,11,6,2147,466,4,

In [0]:
#Aquí arreglamos la inconsistencia de las stats de portero de jugadores que no son porteros
# La condición es: NO es GK Y alguna habilidad GK es > 1
inconsistency_condition = (
    ~col("best_position").contains("GK")
) & (
    (col("gk_diving") > 0)
    | (col("gk_handling") > 0)
    | (col("gk_kicking") > 0)
    | (col("gk_positioning") > 0)
    | (col("gk_reflexes") > 0)
)

#Lista de columnas de habilidades de portero a corregir
gk_columns = [
    "gk_diving",
    "gk_handling",
    "gk_kicking",
    "gk_positioning",
    "gk_reflexes",
]



for column_name in gk_columns:
    df_silver = df_silver.withColumn(
        column_name,
        when(inconsistency_condition, lit(0)).otherwise(col(column_name)),
    )



In [0]:
df = df_silver  # alias


# ============================================================
# 1) COMPLETITUD (no nulos)
# ============================================================

nulls = df.select([
    f.sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).first()

total_nulls = sum(nulls[c] for c in nulls.__fields__)
total_cells = df.count() * len(df.columns)

completitud = 1 - (total_nulls / total_cells)


# ============================================================
# 2) UNICIDAD (sin duplicados)
# ============================================================

total_rows = df.count()
unique_rows = df.dropDuplicates().count()

unicidad = unique_rows / total_rows


# ============================================================
# 3) VALIDEZ (checks moderados y realistas)
# ============================================================

# --- Edad razonable ---
invalid_age = df.filter((col("age") < 15) | (col("age") > 55)).count()

# --- Height válido: debe empezar con números ---
invalid_height = df.filter(
    ~col("height").rlike("^[0-9]{2,3}.*$")
).count()

# --- Weight válido: debe empezar con números ---
invalid_weight = df.filter(
    ~col("weight").rlike("^[0-9]{2,3}.*$")
).count()

# --- Growth razonable (0 a 25 es típico FIFA) ---
invalid_growth = df.filter(
    (col("growth") < 0) | (col("growth") > 25)
).count()

# --- Value / Wage / Release Clause ---
money_columns = ["value", "wage", "release_clause"]

invalid_money = df.filter(
    reduce(
        lambda a, b: a | b,
        [col(c).rlike("^€[0-9MK]+$") for c in money_columns]
    )
).count()

# TOTAL inválidos moderados
invalid_total = (
    invalid_age +
    invalid_height +
    invalid_weight +
    invalid_growth +
    invalid_money
)

# límite para evitar resultados absurdos
invalid_total = min(invalid_total, total_rows)

validez = 1 - (invalid_total / total_rows)
validez = max(validez, 0)   # evitar negativos


# ============================================================
# 4) CONSISTENCIA (coherencia interna)
# ============================================================

inconsistent_gk = df.filter(
    (~col("best_position").contains("GK")) &
    (
        (col("gk_diving") > 0) |
        (col("gk_handling") > 0) |
        (col("gk_kicking") > 0) |
        (col("gk_positioning") > 0) |
        (col("gk_reflexes") > 0)
    )
).count()

consistencia = 1 - (inconsistent_gk / total_rows)


# ============================================================
# 5) EXACTITUD (stats en 0–100)
# ============================================================

stat_columns = [
    'pac', 'sho', 'pas', 'dri', 'def', 'phy',
    'acceleration', 'sprint_speed', 'agility',
    'reactions', 'strength', 'composure'
]

invalid_stats = df.filter(
    reduce(
        lambda a, b: a | b,
        [(col(c) < 0) | (col(c) > 100) for c in stat_columns]
    )
).count()

exactitud = 1 - (invalid_stats / total_rows)


# ============================================================
# 6) ACTUALIDAD (proxy: joined no nulo)
# ============================================================

missing_joined = df.filter(col("joined").isNull()).count()

actualidad = 1 - (missing_joined / total_rows)


# ============================================================
# SCORE GLOBAL
# ============================================================

data_quality_score = (
    completitud +
    unicidad +
    validez +
    consistencia +
    exactitud +
    actualidad
) / 6


# ============================================================
# REPORT FINAL
# ============================================================

print("=== DATA QUALITY REPORT ===")
print(f"Completitud:   {completitud*100:.2f}%")
print(f"Unicidad:      {unicidad*100:.2f}%")
print(f"Validez:       {validez*100:.2f}%")
print(f"Consistencia:  {consistencia*100:.2f}%")
print(f"Exactitud:     {exactitud*100:.2f}%")
print(f"Actualidad:    {actualidad*100:.2f}%")
print("-------------------------------------")
print(f"DATA QUALITY GLOBAL: {data_quality_score*100:.2f}%")


=== DATA QUALITY REPORT ===
Completitud:   99.96%
Unicidad:      100.00%
Validez:       99.98%
Consistencia:  100.00%
Exactitud:     100.00%
Actualidad:    100.00%
-------------------------------------
DATA QUALITY GLOBAL: 99.99%


In [0]:
#metadatos

df_silver = df_silver.withColumn(
    "source_layer", lit("bronze")
).withColumn(
    "processed_timestamp", current_timestamp()
)
display(df_silver.limit(20))

longname,playerurl,nationality,age,overall_rating,potential,id,height,weight,foot,best_overall,best_position,growth,joined,value,wage,release_clause,attacking,crossing,finishing,heading_accuracy,short_passing,volleys,skill,dribbling,curve,fk_accuracy,long_passing,ball_control,movement,acceleration,sprint_speed,agility,reactions,balance,power,shot_power,jumping,stamina,strength,long_shots,mentality,aggression,interceptions,positioning,vision,penalties,composure,defending,marking,standing_tackle,sliding_tackle,goalkeeping,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes,total_stats,base_stats,weak_foot_starss,skills_stars,atack_contribution,defense_contribution,international_reputation,pac,sho,pas,dri,def,phy,hits,team_name,contract,team_original,is_free,contract_raw,num_positions,contract_start_year,years_at_team_2021,value_player,age_category,offensive_score,defensive_score,two_footed,clutch_score,clutch_level,source_layer,processed_timestamp
Lionel Messi,http://sofifa.com/player/158023/lionel-messi/210005/,Argentina,33,93,93,158023,170.2,72.1,Left,93,RW,0,"Jul 1, 2004",67500000,560000,138400000,429,85,95,70,91,88,470,96,93,94,91,96,451,91,80,91,94,95,389,86,68,72,69,94,347,44,40,93,95,75,96,91,32,35,24,54,0,0,0,0,0,2231,466,4,4,Medium,Low,5,85,92,91,95,38,65,372,FC Barcelona,2004 - 2021,FC Barcelona2004 ~ 2021,false,2004 - 2021,3,2004,17,Premium,Veteran,94.17,41.2,1,95.0,Elite,bronze,2026-09-07T20:01:19.058Z
C. Ronaldo dos Santos Aveiro,http://sofifa.com/player/20801/c-ronaldo-dos-santos-aveiro/210005/,Portugal,35,92,92,20801,188.0,83.0,Right,92,ST,0,"Jul 10, 2018",46000000,220000,75900000,437,84,95,90,82,86,414,88,81,76,77,92,431,87,91,87,95,71,444,94,95,84,78,93,353,63,29,95,82,84,95,84,28,32,24,58,0,0,0,0,0,2221,464,4,5,High,Low,5,89,93,81,89,35,77,344,Juventus,2018 - 2022,Juventus2018 ~ 2022,false,2018 - 2022,2,2018,3,High,Late,89.33,39.6,1,90.67,Elite,bronze,2026-09-07T20:01:19.058Z
Jan Oblak,http://sofifa.com/player/200389/jan-oblak/210005/,Slovenia,27,91,93,200389,188.0,87.1,Right,91,GK,2,"Jul 16, 2014",75000000,125000,159400000,95,13,11,15,43,13,109,12,13,14,40,30,307,43,60,67,88,49,268,59,78,41,78,12,140,34,19,11,65,11,68,57,27,12,18,437,87,92,78,90,90,1413,489,3,1,Medium,Medium,3,87,92,78,90,52,90,86,Atlético Madrid,2014 - 2023,Atlético Madrid2014 ~ 2023,false,2014 - 2023,1,2014,7,Premium,Prime,20.17,35.8,0,73.67,Strong,bronze,2026-09-07T20:01:19.058Z
Kevin De Bruyne,http://sofifa.com/player/192985/kevin-de-bruyne/210005/,Belgium,29,91,91,192985,180.3,69.9,Right,91,CAM,0,"Aug 30, 2015",87000000,370000,161000000,407,94,82,55,94,82,441,88,85,83,93,92,398,77,76,78,91,76,408,91,63,89,74,91,408,76,66,88,94,84,91,186,68,65,53,56,0,0,0,0,0,2304,485,5,4,High,High,4,76,86,93,88,64,78,163,Manchester City,2015 - 2023,Manchester City2015 ~ 2023,false,2015 - 2023,2,2015,6,Premium,Veteran,85.83,64.4,1,92.0,Elite,bronze,2026-09-07T20:01:19.058Z
Neymar da Silva Santos Jr.,http://sofifa.com/player/190871/neymar-da-silva-santos-jr/210005/,Brazil,28,91,91,190871,175.3,68.0,Right,91,LW,0,"Aug 3, 2017",90000000,270000,166500000,408,85,87,62,87,87,448,95,88,89,81,95,453,94,89,96,91,83,357,80,62,81,50,84,356,51,36,87,90,92,93,94,35,30,29,59,0,0,0,0,0,2175,451,5,5,High,Medium,5,91,85,86,94,36,59,273,Paris Saint-Germain,2017 - 2022,Paris Saint-Germain2017 ~ 2022,false,2017 - 2022,2,2017,4,Premium,Prime,90.5,36.2,1,91.33,Elite,bronze,2026-09-07T20:01:19.058Z
Robert Lewandowski,http://sofifa.com/player/188545/robert-lewandowski/210005/,Poland,31,91,91,188545,182.9,79.8,Right,91,ST,0,"Jul 1, 2014",80000000,240000,132000000,423,71,94,85,84,89,407,85,79,85,70,88,407,77,78,77,93,82,420,89,84,76,86,85,391,81,49,94,79,88,88,96,35,42,19,51,0,0,0,0,0,2195,457,4,4,High,Medium,4,78,91,78,85,43,82,182,FC Bayern München,2014 - 2023,FC Bayern München2014 ~ 2023,false,2014 - 2023,1,2014,7,Premium,Veteran,84.67,47.8,1,86.67,Elite,bronze,2026-09-07T20:01:19.058Z
Kylian Mbappé,http://sofifa.com/player/231747/kylian-mbappe/210005/,France,21,90,95,23

In [0]:
#guardamos en silver
df_silver.write.mode("overwrite").saveAsTable("fifa_catalog.silver.df_silver")

In [0]:
spark.sql("""
OPTIMIZE fifa_catalog.silver.df_silver
ZORDER BY (id, nationality, longname)
""")



DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
spark.sql("""
VACUUM fifa_catalog.silver.df_silver RETAIN 168 HOURS
""")

DataFrame[path: string]

## Fin del notebook

La tabla `fifa_catalog.silver.df_silver` ya está escrita, optimizada y con las métricas de calidad recalculadas. Continúa con `MODELLING.ipynb` para construir la capa Gold.